# Prerequisites

In [0]:
from pyspark.sql.functions import *
customers_df = spark.read.table("ecommerce.silver.customers")
orders_df = spark.read.table("ecommerce.silver.orders")
# order_items_df = spark.read.table("ecommerce.silver.order_items")
# products_df = spark.read.table("ecommerce.silver.products")
# payments_df = spark.read.table("ecommerce.silver.payments")
# inventory_df = spark.read.table("ecommerce.silver.inventory")
clickstream_df = spark.read.table("ecommerce.silver.clickstream")

# Calculations

## Join customers, clickstream and orders

In [0]:
cust_df = customers_df.join(clickstream_df, "customer_id", "left" ).join(orders_df, "customer_id", "left").withColumn("customer_name", concat(col("first_name"), lit(" "), col("last_name")))
cust_df.limit(5).display()

In [0]:
cust_df.select("event_type").distinct().display()

## sessions, product_views, add_to_cart_events, checkout_events, purchases

In [0]:
cust_beh_df = cust_df.groupBy("customer_id", "customer_name").agg(
    countDistinct("session_id").alias("sessions"),
    countDistinct(
        when(col("event_type") == lit("PRODUCT_VIEW"), col("session_id"))
    ).alias("product_views"),
    countDistinct(
        when(col("event_type") == lit("ADD_TO_CART"), col("session_id"))
    ).alias("add_to_cart_events"),
    countDistinct(
        when(col("event_type") == lit("CHECKOUT"), col("session_id"))
    ).alias("checkout_events"),
    countDistinct(
        when(col("event_type") == lit("PURCHASE"), col("session_id"))
    ).alias("purchases"),
)
cust_beh_df.limit(5).display()

## Calculate Rate - view_to_cart_rate, cart_to_checkout_rate, checkout_to_purchase_rate, overall_conversion_rate 

In [0]:
cust_beh_rate = cust_beh_df.withColumn(
    "view_to_cart_rate",
    round(try_divide(col("add_to_cart_events"), col("product_views")), 2),
).withColumn(
    "cart_to_checkout_rate",
    round(try_divide(col("checkout_events"), col("add_to_cart_events")), 2),
).withColumn(
    "checkout_to_purchase_rate",
    round(try_divide(col("purchases"), col("checkout_events")), 2),
).withColumn(
    "overall_conversion_rate",
    round(try_divide(col("purchases"), col("product_views")), 2),
)
cust_beh_rate.limit(5).display()

# Create gold.customer_behavior Delta Table

In [0]:
cust_beh_rate.write.mode("overwrite").option("mergeSchema","true").saveAsTable("ecommerce.gold.customer_behavior")
spark.read.table("ecommerce.gold.customer_behavior").limit(5).display()